# 2. Rule-Based Phishing Detection Model

## 1. Foundation & Approach

### 1.1 Business Context

**Business Requirement:**
A small security consulting firm needs a way for their staff (security analysts and administrative personnel) to quickly determine whether a URL is phishing or legitimate. Staff currently evaluate suspicious URLs from emails manually, which is:
- Time-consuming (each URL takes 5-10 minutes to investigate)
- Inconsistent (depends on analyst experience)
- Not explainable to clients ("I just have a bad feeling about this URL")

**What they need:** A tool that says "This is phishing because X, Y, Z" with concrete reasons.

**What We Discovered - The Phishing Pattern:**
Our exploration of 235,795 URLs (42.8% phishing, 57.2% legitimate) from dataset4 revealed that **phishing sites are fundamentally simple**:

**The Phishing Formula (based on dataset4):**
1. **Zero resources** - 29.4% have no JavaScript, CSS, or images (69,212 URLs)
2. **No encryption** - 50.78% of phishing URLs use HTTP instead of HTTPS
3. **Minimal code** - 27.4% have zero complexity (64,486 URLs with bare HTML forms)
4. **No trust signals** - 13.3% lack basic indicators (31,407 URLs missing title, favicon, description, copyright)
5. **No professional standards** - 93.5% lack robots.txt, responsive design, or social media integration (71,930 URLs)

**Why this pattern exists:**
Phishing is **minimal effort credential harvesting**:
- Copy a login form (PayPal, Gmail, Bank)
- Host on cheap domain (.tk, .ml, .ga)
- POST form data to attacker's server
- Lives for hours/days, disappears before takedown

**Legitimate sites are the opposite:**
- Rich resources (JavaScript frameworks, CSS libraries, images)
- Professional standards (responsive, social media, SEO)
- Trust indicators (metadata, favicons, copyright)
- Code complexity (thousands of lines of JavaScript)

**Why Rule-Based Detection Works:**
These patterns are **not subtle** - they're obvious red flags a human would spot. A rule-based system can:
- Explain decisions clearly ("No HTTPS + zero resources + no trust signals = phishing")
- Be transparent (staff can verify rules make sense)
- Be maintainable (update rules as phishing tactics evolve)

**This Notebook:**
Build a proof-of-concept using these behavioral patterns, tested on historical data from dataset4. This is a **research prototype** to validate the approach, not a production-ready system.

### 1.2 Dataset Reality Check - Bias & Limitations

**The Circular Validation Problem:**

We are using dataset4 to both:
1. Discover the rules (e.g., "ResourceTypeScore = 0 → phishing")  
2. Test the rules (measure accuracy on the same dataset)

**What this means:**
- Accuracy on dataset4 shows the rules work on **this historical data**
- It does NOT prove the rules work on new, unseen phishing
- This is a proof-of-concept, not a production validation

**What we acknowledge:**
- No external validation dataset available
- Cannot claim generalization without out-of-sample testing
- Performance on real-world traffic is unknown

### 1.3 Why URL Patterns Alone Fail

**What We Found in Exploration:**

Some URL-based features are **strong indicators**:
- TLD = .top → 99.9% phishing (2,329 URLs)
- TLD = .edu → 99.7% legitimate (1,861 URLs)
- IsDomainIP = 1 → 100% phishing
- IsHTTPS = 0 → 100% phishing (50.78% of phishing caught)

These work because they're **structural absolutes** (educational institutions control .edu, IP addresses are suspicious).

**Where URL Patterns Fall Short:**

Many URL features are **correlations, not guarantees**:
- DomainLength > 30 → 92.2% phishing (but legitimate sites can have long domains)
- High digit count → strong signal (but api2024.example.io could be legitimate)
- Certain TLDs (.io, .co) → mixed (89.7% phishing for .io, but many legitimate startups use it)

URL-only detection misses the full picture.

**The Missing Piece: Webpage Behavior**

Even if a URL looks suspicious, the **webpage itself reveals the truth**:
- Does it have JavaScript, CSS, images? (29.4% of phishing have ZERO)
- Trust indicators present? (13.3% of phishing have NONE)
- Code complexity? (27.4% of phishing = bare HTML)
- Professional standards? (93.5% of phishing lack them)

**Why Both Matter:**

URL analysis catches obvious cases fast (no HTTPS, .top domain, IP address).  
Behavioral analysis catches sophisticated phishing (suspicious URL + simple webpage = phishing).

**Our Approach:**

Use URL features where they're strong, but **fetch and analyze the webpage** to get the complete picture.

### 1.4 The Real Signal - Behavioral Analysis

**What Behavioral Features Measure:**

Dataset4's 56 features can be grouped by what they evaluate:

**1. Resource Investment** (NoOfJS, NoOfCSS, NoOfImage)
- Measures: Developer effort, time investment
- Why it matters: Building a React app takes weeks; copying a login form takes minutes

**2. Trust Signals** (HasTitle, HasFavicon, HasDescription, HasCopyrightInfo)
- Measures: Attention to user experience, brand presence
- Why it matters: Legitimate businesses care about metadata; phishers copy-paste HTML

**3. Code Complexity** (LineOfCode, LargestLineLength)
- Measures: Functional depth vs static content
- Why it matters: Real sites have logic; phishing sites have forms that POST to external servers

**4. Professional Standards** (Robots, IsResponsive, HasSocialNet)
- Measures: Long-term business presence, SEO, user engagement
- Why it matters: Phishing sites live for hours/days; no point in responsive design or social media

**The Framework:**

These aren't random features - they're measuring **investment signals**:
- High investment (time, money, expertise) → likely legitimate
- Zero investment (bare minimum to steal credentials) → likely phishing

**Why This Generalizes (Hypothesis):**

Phishing economics don't change:
- Short lifespan → no ROI on professional development
- High volume, low success rate → minimal effort per site
- Disposable infrastructure → why build quality?

**What We're Testing:**

Can we detect phishing by measuring **effort invested in the webpage**?

### 1.5 Our Architecture - Live Website Analyzer

**The System Flow:**

```
URL input
    ↓
Fetch webpage (HTTP request, 5s timeout)
    ↓
Parse HTML content
    ↓
Extract relevant features
    ↓
Apply rules (covered in later sections)
    ↓
Output: verdict + confidence + explanation
```

**What Gets Extracted:**

**From the URL itself (fast, no fetch required):**
- Domain, TLD, length metrics
- Character counts (letters, digits, special chars)
- URL structure (subdomains, path, query params)

**From the webpage (requires fetch):**
- **Resource counts:** JavaScript files, CSS files, images
- **HTML metadata:** Title, favicon, description, copyright
- **Code metrics:** Lines of code, complexity measures
- **Professional indicators:** Robots.txt, responsive design, social media links
- **Security features:** HTTPS status
- **References:** Internal vs external links

**Why Fetching is Required:**

Dataset4 provides pre-extracted features. In production, we must:
1. Make HTTP request to the URL
2. Download HTML content
3. Parse and extract features needed for our rules
4. Then apply detection logic

**This is webpage analysis, not URL string matching.**

### 1.6 Scope of This Notebook

**What We're Building:**

**Phase 1: Feature Selection**
- Identify strongest URL features (IsHTTPS, IsDomainIP, TLD)
- Identify strongest behavioral features (ResourceTypeScore, TrustScore, ProfessionalScore)
- Justify selections based on exploration findings from dataset4

**Phase 2: Rule-Based Model**
- Build cascading rules using selected features
- Evaluate on dataset4 (train/test split)
- Measure: accuracy, precision, recall, false positive/negative rates
- Identify where rules succeed and where they fail

**Phase 3: ML Extension (Conditional)**
- **IF** rules show clear weaknesses (e.g., >5% false positive rate)
- **THEN** build ML model for edge cases
- Compare rule-based vs hybrid approach
- Maintain explainability requirement

**Decision Point:** ML is added **only if rules prove insufficient**. We test rules first.

**What's NOT in This Notebook:**
- Production data extraction tool (requires live URL fetching - dataset4 has pre-extracted features)
- Deployment architecture (API, scaling, monitoring)
- Continuous learning (feedback loops, model retraining)

**Success Criteria:**
- Clear feature selection rationale
- Rule-based model with documented performance
- Decision on whether ML is needed
- Explainable predictions for business requirement